**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**


# Báo cáo bài tập về nhà Buổi 13: Constraint Satisfaction Problem

Trong buổi học này, tôi học nhóm bài toán **CSP - Constraint Satisfaction Problem**. Khác với các buổi trước, nơi tôi thường biểu diễn bài toán dưới dạng tìm đường đi trong không gian trạng thái, CSP tập trung vào việc **gán giá trị cho các biến sao cho mọi ràng buộc đều được thỏa mãn**.

Một bài toán CSP thường được mô tả bởi ba thành phần:

- **Variables**: tập biến cần gán giá trị.
- **Domains**: miền giá trị có thể nhận của từng biến.
- **Constraints**: các ràng buộc giữa biến, cho biết tổ hợp giá trị nào là hợp lệ.

Ví dụ trong bài toán tô màu bản đồ, biến là các vùng lãnh thổ, domain là các màu, constraint là hai vùng kề nhau không được trùng màu.

## 1. Backtracking Search

Backtracking là cách giải CSP cơ bản nhất. Tôi gán giá trị cho từng biến theo thứ tự. Nếu phát hiện một gán trị làm vi phạm ràng buộc, tôi quay lui để thử giá trị khác.

Tư tưởng chính:

```text
Chọn một biến chưa gán
Thử từng giá trị trong domain
Nếu không vi phạm constraint thì đi tiếp
Nếu không còn giá trị hợp lệ thì backtrack
```

Backtracking đầy đủ với CSP hữu hạn, nhưng có thể chậm vì nó chỉ phát hiện lỗi sau khi đã gán thử.

## 2. Forward Checking

Forward Checking cải tiến backtracking bằng cách nhìn trước một bước. Khi tôi gán `X = value`, thuật toán sẽ xóa khỏi domain của các biến chưa gán những giá trị không còn tương thích với `X`.

Tư tưởng chính:

```text
Sau mỗi phép gán, cập nhật domain của các biến liên quan
Nếu một biến nào đó bị rỗng domain thì dừng nhánh sớm
```

Nhờ vậy, Forward Checking phát hiện nhánh sai sớm hơn backtracking thường.

## 3. Arc Consistency - AC-3

Arc Consistency mạnh hơn Forward Checking vì nó không chỉ kiểm tra từ biến vừa gán sang biến lân cận, mà lan truyền ràng buộc qua các cung trong đồ thị CSP.

Một cung `(Xi, Xj)` là nhất quán nếu với mỗi giá trị `x` trong domain của `Xi`, tồn tại ít nhất một giá trị `y` trong domain của `Xj` sao cho ràng buộc giữa `Xi` và `Xj` được thỏa mãn.

Tư tưởng chính:

```text
Đưa các cung vào queue
Nếu domain của Xi bị cắt, các cung liên quan tới Xi phải được xét lại
Lặp đến khi không còn domain nào thay đổi
```

AC-3 có thể dùng trước backtracking hoặc dùng sau mỗi phép gán để giảm domain mạnh hơn.

## 4. Min-Conflicts

Min-Conflicts là thuật toán local search cho CSP. Thay vì xây dựng assignment từng phần, tôi tạo một assignment đầy đủ ngay từ đầu, sau đó sửa dần các biến đang gây conflict.

Tư tưởng chính:

```text
Tạo assignment đầy đủ ngẫu nhiên
Chọn một biến đang conflict
Đổi giá trị của biến đó sang giá trị làm số conflict nhỏ nhất
Lặp đến khi không còn conflict
```

Min-Conflicts thường chạy tốt với CSP lớn, nhưng không đảm bảo luôn tìm được nghiệm nếu bị kẹt hoặc vượt quá số bước cho phép.

## 5. Hai bài toán minh họa

Trong notebook này, tôi minh họa CSP bằng hai bài toán:

1. **Tô màu bản đồ**: đây là ví dụ kinh điển của CSP. Biến là các vùng trên bản đồ Australia, domain là ba màu, constraint là hai vùng kề nhau không được trùng màu.
2. **8-puzzle dưới góc nhìn CSP**: tôi không dùng 8-puzzle để tìm đường đi như BFS/A*/IDA*. Ở đây, tôi xem mỗi ô trên bảng 3x3 là một biến, domain là các tile `0..8`, constraint chính là các ô không được trùng tile. Một vài ô được cố định như gợi ý để bài toán có ý nghĩa trực quan.

Cách mô hình hóa 8-puzzle này không thay thế thuật toán tìm kiếm đường đi. Nó chỉ giúp tôi nhìn 8-puzzle như một bài toán thỏa mãn ràng buộc: điền các tile vào bảng sao cho đúng các constraint đã đặt ra.

## 6. So sánh ngắn gọn

| Thuật toán | Cách làm | Điểm mạnh | Hạn chế |
|---|---|---|---|
| Backtracking | Gán từng biến và quay lui khi sai | Đơn giản, đầy đủ | Dễ thử nhiều nhánh sai |
| Forward Checking | Gán biến rồi cắt domain biến liên quan | Phát hiện lỗi sớm | Chỉ nhìn trước cục bộ |
| AC-3 | Duy trì nhất quán cung trên đồ thị constraint | Cắt domain mạnh hơn | Tốn thêm chi phí lan truyền |
| Min-Conflicts | Sửa assignment đầy đủ bằng local search | Thường nhanh với bài toán lớn | Không đảm bảo tối ưu/đầy đủ trong số bước hữu hạn |


In [ ]:
# =========================================================================
# BUỔI 13 - CSP: BACKTRACKING, FORWARD CHECKING, AC-3, MIN-CONFLICTS
# Minh họa trên Map Coloring và 8-Puzzle dạng CSP
# =========================================================================

from dataclasses import dataclass
import random
import tkinter as tk
from tkinter import ttk


MAP_COLORS = ("Red", "Green", "Blue")
COLOR_HEX = {
    "Red": "#f87171",
    "Green": "#86efac",
    "Blue": "#93c5fd",
    0: "#bfdbfe",
    1: "#ffffff",
    2: "#ffffff",
    3: "#ffffff",
    4: "#ffffff",
    5: "#ffffff",
    6: "#ffffff",
    7: "#ffffff",
    8: "#ffffff",
    "?": "#f3f4f6",
}


@dataclass
class CSP:
    name: str
    variables: list
    domains: dict
    neighbors: dict
    constraint: object
    positions: dict
    kind: str
    fixed: set


def make_map_coloring_csp():
    variables = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
    edges = [
        ("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"),
        ("SA", "Q"), ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"),
        ("NSW", "V"),
    ]
    neighbors = {var: set() for var in variables}
    for a, b in edges:
        neighbors[a].add(b)
        neighbors[b].add(a)

    positions = {
        "WA": (80, 150), "NT": (210, 90), "SA": (225, 210),
        "Q": (365, 120), "NSW": (385, 270), "V": (320, 360), "T": (410, 420),
    }
    domains = {var: list(MAP_COLORS) for var in variables}

    def constraint(_a, value_a, _b, value_b):
        return value_a != value_b

    return CSP(
        name="Map Coloring - Australia",
        variables=variables,
        domains=domains,
        neighbors={var: sorted(items) for var, items in neighbors.items()},
        constraint=constraint,
        positions=positions,
        kind="map",
        fixed=set(),
    )


def make_puzzle_csp():
    variables = [f"C{i}" for i in range(9)]
    fixed_values = {
        "C0": 1,
        "C1": 2,
        "C2": 3,
        "C4": 0,
    }
    domains = {}
    for var in variables:
        domains[var] = [fixed_values[var]] if var in fixed_values else list(range(9))

    neighbors = {var: [other for other in variables if other != var] for var in variables}
    positions = {f"C{r * 3 + c}": (c, r) for r in range(3) for c in range(3)}

    def constraint(_a, value_a, _b, value_b):
        return value_a != value_b

    return CSP(
        name="8-Puzzle as CSP",
        variables=variables,
        domains=domains,
        neighbors=neighbors,
        constraint=constraint,
        positions=positions,
        kind="puzzle",
        fixed=set(fixed_values),
    )


def copy_domains(domains):
    return {var: list(values) for var, values in domains.items()}


def assignment_conflicts(csp, assignment):
    conflicts = []
    for var in csp.variables:
        if var not in assignment:
            continue
        for other in csp.neighbors[var]:
            if other not in assignment or var > other:
                continue
            if not csp.constraint(var, assignment[var], other, assignment[other]):
                conflicts.append((var, other))
    return conflicts


def is_consistent(csp, var, value, assignment):
    for other in csp.neighbors[var]:
        if other in assignment and not csp.constraint(var, value, other, assignment[other]):
            return False
    return True


def select_unassigned_variable(csp, assignment, domains):
    remaining = [var for var in csp.variables if var not in assignment]
    return min(remaining, key=lambda var: (len(domains[var]), -len(csp.neighbors[var]), var))


def order_domain_values(csp, var, domains, assignment):
    def removed_count(value):
        total = 0
        for other in csp.neighbors[var]:
            if other in assignment:
                continue
            for other_value in domains[other]:
                if not csp.constraint(var, value, other, other_value):
                    total += 1
        return total

    return sorted(domains[var], key=lambda value: (removed_count(value), str(value)))


def forward_check(csp, var, value, domains, assignment):
    new_domains = copy_domains(domains)
    new_domains[var] = [value]
    removed = []
    for other in csp.neighbors[var]:
        if other in assignment:
            continue
        for other_value in list(new_domains[other]):
            if not csp.constraint(var, value, other, other_value):
                new_domains[other].remove(other_value)
                removed.append((other, other_value))
        if not new_domains[other]:
            return False, new_domains, removed
    return True, new_domains, removed


def revise(csp, domains, xi, xj):
    removed = []
    for x in list(domains[xi]):
        has_support = any(csp.constraint(xi, x, xj, y) for y in domains[xj])
        if not has_support:
            domains[xi].remove(x)
            removed.append((xi, x))
    return removed


def ac3(csp, domains):
    queue = [(xi, xj) for xi in csp.variables for xj in csp.neighbors[xi]]
    removed = []
    processed = 0
    while queue:
        xi, xj = queue.pop(0)
        processed += 1
        revised = revise(csp, domains, xi, xj)
        if revised:
            removed.extend(revised)
            if not domains[xi]:
                return False, removed, processed
            for xk in csp.neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True, removed, processed


def make_step(csp, assignment, domains, status, message, counters, current_var=None, current_value=None):
    conflicts = assignment_conflicts(csp, assignment)
    return {
        "csp": csp,
        "assignment": dict(assignment),
        "domains": copy_domains(domains),
        "status": status,
        "message": message,
        "current_var": current_var,
        "current_value": current_value,
        "checks": counters.get("checks", 0),
        "backtracks": counters.get("backtracks", 0),
        "pruned": counters.get("pruned", 0),
        "steps": counters.get("steps", 0),
        "conflicts": conflicts,
    }


def backtracking_solver(csp, inference="none"):
    counters = {"checks": 0, "backtracks": 0, "pruned": 0, "steps": 0}
    start_domains = copy_domains(csp.domains)

    if inference == "ac3":
        ok, removed, processed = ac3(csp, start_domains)
        counters["pruned"] += len(removed)
        counters["steps"] += 1
        yield make_step(
            csp, {}, start_domains, "AC-3 khởi tạo",
            f"AC-3 xử lý {processed} cung và cắt {len(removed)} giá trị.",
            counters,
        )
        if not ok:
            yield make_step(csp, {}, start_domains, "Thất bại", "Domain bị rỗng sau AC-3.", counters)
            return

    def backtrack(assignment, domains, depth):
        if len(assignment) == len(csp.variables):
            counters["steps"] += 1
            yield make_step(csp, assignment, domains, "Thành công", "Tất cả biến đã được gán hợp lệ.", counters)
            return True

        var = select_unassigned_variable(csp, assignment, domains)
        counters["steps"] += 1
        yield make_step(csp, assignment, domains, "Chọn biến", f"MRV chọn biến {var}.", counters, var)

        for value in order_domain_values(csp, var, domains, assignment):
            counters["checks"] += 1
            if not is_consistent(csp, var, value, assignment):
                counters["steps"] += 1
                yield make_step(
                    csp, assignment, domains, "Loại giá trị",
                    f"{var}={value} vi phạm ràng buộc với assignment hiện tại.",
                    counters, var, value,
                )
                continue

            next_assignment = dict(assignment)
            next_assignment[var] = value
            next_domains = copy_domains(domains)
            next_domains[var] = [value]
            counters["steps"] += 1
            yield make_step(
                csp, next_assignment, next_domains, "Thử gán",
                f"Thử gán {var}={value}.",
                counters, var, value,
            )

            ok = True
            if inference in ("forward", "ac3"):
                ok, next_domains, removed = forward_check(csp, var, value, next_domains, next_assignment)
                counters["pruned"] += len(removed)
                counters["steps"] += 1
                yield make_step(
                    csp, next_assignment, next_domains,
                    "Forward Checking" if ok else "Domain rỗng",
                    f"Forward checking cắt {len(removed)} giá trị sau {var}={value}.",
                    counters, var, value,
                )
            if ok and inference == "ac3":
                ok, removed, processed = ac3(csp, next_domains)
                counters["pruned"] += len(removed)
                counters["steps"] += 1
                yield make_step(
                    csp, next_assignment, next_domains,
                    "AC-3 lan truyền" if ok else "Domain rỗng",
                    f"AC-3 xử lý {processed} cung và cắt thêm {len(removed)} giá trị.",
                    counters, var, value,
                )

            if ok:
                solved = yield from backtrack(next_assignment, next_domains, depth + 1)
                if solved:
                    return True

            counters["backtracks"] += 1
            counters["steps"] += 1
            yield make_step(
                csp, assignment, domains, "Backtrack",
                f"Quay lui khỏi nhánh {var}={value}.",
                counters, var, value,
            )

        return False

    solved = yield from backtrack({}, start_domains, 0)
    if not solved:
        counters["steps"] += 1
        yield make_step(csp, {}, start_domains, "Thất bại", "Không tìm được assignment hợp lệ.", counters)


def min_conflicts_solver(csp, max_steps=250):
    counters = {"checks": 0, "backtracks": 0, "pruned": 0, "steps": 0}
    assignment = {}
    for var in csp.variables:
        assignment[var] = csp.domains[var][0] if var in csp.fixed else random.choice(csp.domains[var])
    domains = copy_domains(csp.domains)

    yield make_step(csp, assignment, domains, "Khởi tạo", "Tạo assignment đầy đủ ngẫu nhiên.", counters)

    for step in range(1, max_steps + 1):
        counters["steps"] = step
        conflicts = assignment_conflicts(csp, assignment)
        if not conflicts:
            yield make_step(csp, assignment, domains, "Thành công", "Không còn conflict.", counters)
            return

        conflicted_vars = sorted({var for pair in conflicts for var in pair if var not in csp.fixed})
        if not conflicted_vars:
            yield make_step(csp, assignment, domains, "Kẹt", "Conflict chỉ còn ở biến cố định.", counters)
            return

        var = random.choice(conflicted_vars)

        def conflict_count(value):
            total = 0
            trial = dict(assignment)
            trial[var] = value
            for other in csp.neighbors[var]:
                if other in trial and not csp.constraint(var, value, other, trial[other]):
                    total += 1
            return total

        scores = [(value, conflict_count(value)) for value in csp.domains[var]]
        best_score = min(score for _, score in scores)
        best_values = [value for value, score in scores if score == best_score]
        value = random.choice(best_values)
        old_value = assignment[var]
        assignment[var] = value
        counters["checks"] += len(scores)

        yield make_step(
            csp, assignment, domains, "Min-Conflicts",
            f"Chọn biến conflict {var}: {old_value} -> {value}, conflict cục bộ nhỏ nhất = {best_score}.",
            counters, var, value,
        )

    yield make_step(csp, assignment, domains, "Dừng", "Đạt giới hạn bước của Min-Conflicts.", counters)


def make_runner(csp, algorithm):
    if algorithm == "Backtracking":
        return backtracking_solver(csp, inference="none")
    if algorithm == "Forward Checking":
        return backtracking_solver(csp, inference="forward")
    if algorithm == "AC-3 + Backtracking":
        return backtracking_solver(csp, inference="ac3")
    return min_conflicts_solver(csp)


class CSPApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Buổi 13 - Constraint Satisfaction Problem")
        self.root.geometry("1180x760")
        self.root.minsize(1080, 680)

        self.problem_var = tk.StringVar(value="Map Coloring")
        self.algorithm_var = tk.StringVar(value="Backtracking")
        self.speed_var = tk.IntVar(value=650)
        self.runner = None
        self.current_step = None
        self.auto_job = None

        self._build_style()
        self._build_ui()
        self.reset()

    def _build_style(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass
        style.configure("TFrame", background="#f5f7fb")
        style.configure("Panel.TFrame", background="white", relief="solid", borderwidth=1)
        style.configure("TLabel", background="#f5f7fb", font=("Segoe UI", 10))
        style.configure("Title.TLabel", font=("Segoe UI", 17, "bold"), foreground="#111827")
        style.configure("Subtitle.TLabel", font=("Segoe UI", 10), foreground="#4b5563")
        style.configure("PanelTitle.TLabel", background="white", font=("Segoe UI", 11, "bold"))
        style.configure("TButton", font=("Segoe UI", 10))

    def _build_ui(self):
        main = ttk.Frame(self.root, padding=16)
        main.pack(fill="both", expand=True)

        ttk.Label(main, text="Buổi 13 - Constraint Satisfaction Problem", style="Title.TLabel").pack(anchor="w")
        ttk.Label(
            main,
            text="Backtracking, Forward Checking, AC-3 và Min-Conflicts trên Map Coloring và 8-Puzzle CSP",
            style="Subtitle.TLabel",
        ).pack(anchor="w", pady=(2, 12))

        controls = ttk.Frame(main)
        controls.pack(fill="x", pady=(0, 12))

        ttk.Label(controls, text="Bài toán").pack(side="left")
        problem_box = ttk.Combobox(
            controls,
            textvariable=self.problem_var,
            values=["Map Coloring", "8-Puzzle CSP"],
            state="readonly",
            width=18,
        )
        problem_box.pack(side="left", padx=(6, 16))
        problem_box.bind("<<ComboboxSelected>>", lambda _event: self.reset())

        ttk.Label(controls, text="Thuật toán").pack(side="left")
        algorithm_box = ttk.Combobox(
            controls,
            textvariable=self.algorithm_var,
            values=["Backtracking", "Forward Checking", "AC-3 + Backtracking", "Min-Conflicts"],
            state="readonly",
            width=24,
        )
        algorithm_box.pack(side="left", padx=(6, 16))
        algorithm_box.bind("<<ComboboxSelected>>", lambda _event: self.reset())

        ttk.Button(controls, text="Reset", command=self.reset).pack(side="left", padx=3)
        ttk.Button(controls, text="Next Step", command=self.next_step).pack(side="left", padx=3)
        ttk.Button(controls, text="Auto Run", command=self.auto_run).pack(side="left", padx=3)
        ttk.Button(controls, text="Stop", command=self.stop_auto).pack(side="left", padx=3)
        ttk.Label(controls, text="Tốc độ").pack(side="left", padx=(18, 4))
        ttk.Scale(controls, from_=150, to=1400, variable=self.speed_var, orient="horizontal", length=140).pack(side="left")

        body = ttk.Frame(main)
        body.pack(fill="both", expand=True)

        left = ttk.Frame(body, style="Panel.TFrame", padding=12)
        left.pack(side="left", fill="both", expand=True, padx=(0, 10))
        right = ttk.Frame(body, style="Panel.TFrame", padding=12)
        right.pack(side="right", fill="both", expand=False)

        ttk.Label(left, text="Không gian CSP", style="PanelTitle.TLabel").pack(anchor="w")
        self.canvas = tk.Canvas(left, width=690, height=560, bg="white", highlightthickness=0)
        self.canvas.pack(fill="both", expand=True, pady=(10, 0))

        ttk.Label(right, text="Thông tin bước chạy", style="PanelTitle.TLabel").pack(anchor="w")
        self.metrics_text = tk.Text(right, width=42, height=20, font=("Consolas", 10), bg="#ffffff", relief="flat")
        self.metrics_text.pack(fill="x", pady=(10, 10))
        self.metrics_text.configure(state="disabled")

        ttk.Label(right, text="Log", style="PanelTitle.TLabel").pack(anchor="w")
        self.log_text = tk.Text(right, width=42, height=18, font=("Consolas", 9), bg="#111827", fg="#e5e7eb", relief="flat")
        self.log_text.pack(fill="both", expand=True, pady=(10, 0))
        self.log_text.configure(state="disabled")

    def make_csp(self):
        return make_map_coloring_csp() if self.problem_var.get() == "Map Coloring" else make_puzzle_csp()

    def reset(self):
        self.stop_auto()
        csp = self.make_csp()
        self.runner = make_runner(csp, self.algorithm_var.get())
        self.clear_log()
        self.next_step()

    def next_step(self):
        if self.runner is None:
            self.reset()
            return False
        try:
            self.current_step = next(self.runner)
            self.render(self.current_step)
            self.append_log(self.current_step["message"])
            if self.current_step["status"] in ("Thành công", "Thất bại", "Dừng", "Kẹt"):
                self.stop_auto()
            return True
        except StopIteration:
            self.append_log("Thuật toán đã dừng.")
            self.stop_auto()
            return False

    def auto_run(self):
        self.stop_auto()

        def tick():
            if self.next_step():
                self.auto_job = self.root.after(int(self.speed_var.get()), tick)

        tick()

    def stop_auto(self):
        if self.auto_job is not None:
            self.root.after_cancel(self.auto_job)
            self.auto_job = None

    def render(self, step):
        self.canvas.delete("all")
        csp = step["csp"]
        if csp.kind == "map":
            self.draw_map(step)
        else:
            self.draw_puzzle(step)
        self.render_metrics(step)

    def draw_map(self, step):
        csp = step["csp"]
        assignment = step["assignment"]
        domains = step["domains"]
        for var in csp.variables:
            x, y = csp.positions[var]
            color = assignment.get(var)
            fill = COLOR_HEX.get(color, "#f9fafb")
            outline = "#ef4444" if any(var in pair for pair in step["conflicts"]) else "#111827"
            self.canvas.create_rectangle(x, y, x + 95, y + 58, fill=fill, outline=outline, width=3)
            self.canvas.create_text(x + 47, y + 20, text=var, font=("Segoe UI", 12, "bold"), fill="#111827")
            self.canvas.create_text(
                x + 47, y + 42,
                text=str(color) if color else "/".join(domains[var]),
                font=("Segoe UI", 8),
                fill="#374151",
            )

        for var in csp.variables:
            x1, y1 = csp.positions[var]
            for other in csp.neighbors[var]:
                if var < other:
                    x2, y2 = csp.positions[other]
                    self.canvas.create_line(x1 + 47, y1 + 29, x2 + 47, y2 + 29, fill="#9ca3af", width=2)

    def draw_puzzle(self, step):
        csp = step["csp"]
        assignment = step["assignment"]
        domains = step["domains"]
        cell = 118
        start_x, start_y = 145, 70
        for index, var in enumerate(csp.variables):
            col = index % 3
            row = index // 3
            x1 = start_x + col * cell
            y1 = start_y + row * cell
            value = assignment.get(var, "?")
            fill = "#dbeafe" if value == 0 else ("#e0f2fe" if var in csp.fixed else "#ffffff")
            outline = "#ef4444" if any(var in pair for pair in step["conflicts"]) else "#111827"
            self.canvas.create_rectangle(x1, y1, x1 + cell - 8, y1 + cell - 8, fill=fill, outline=outline, width=3)
            self.canvas.create_text(
                x1 + cell / 2 - 4, y1 + 42,
                text="" if value == 0 else str(value),
                font=("Segoe UI", 26, "bold"),
                fill="#111827",
            )
            domain_text = "fixed" if var in csp.fixed else "D={" + ",".join(str(v) for v in domains[var][:6]) + ("..." if len(domains[var]) > 6 else "") + "}"
            self.canvas.create_text(x1 + cell / 2 - 4, y1 + 82, text=domain_text, font=("Segoe UI", 8), fill="#4b5563")

    def render_metrics(self, step):
        assignment_lines = [f"{var}={value}" for var, value in step["assignment"].items()]
        domain_lines = []
        for var in step["csp"].variables:
            values = step["domains"][var]
            domain_lines.append(f"{var}: {values}")
        conflict_text = ", ".join(f"{a}-{b}" for a, b in step["conflicts"]) or "-"

        lines = [
            f"Problem   : {step['csp'].name}",
            f"Algorithm : {self.algorithm_var.get()}",
            f"Status    : {step['status']}",
            f"Variable  : {step.get('current_var') or '-'}",
            f"Value     : {step.get('current_value') if step.get('current_value') is not None else '-'}",
            f"Steps     : {step['steps']}",
            f"Checks    : {step['checks']}",
            f"Pruned    : {step['pruned']}",
            f"Backtrack : {step['backtracks']}",
            f"Conflicts : {conflict_text}",
            "",
            "Assignment:",
            *(assignment_lines or ["-"]),
            "",
            "Domains:",
            *domain_lines,
        ]

        self.metrics_text.configure(state="normal")
        self.metrics_text.delete("1.0", "end")
        self.metrics_text.insert("end", "\n".join(lines))
        self.metrics_text.configure(state="disabled")

    def append_log(self, text):
        self.log_text.configure(state="normal")
        self.log_text.insert("end", text + "\n")
        self.log_text.see("end")
        self.log_text.configure(state="disabled")

    def clear_log(self):
        self.log_text.configure(state="normal")
        self.log_text.delete("1.0", "end")
        self.log_text.configure(state="disabled")


if __name__ == "__main__":
    root = tk.Tk()
    app = CSPApp(root)
    root.mainloop()
